<a href="https://colab.research.google.com/github/RDGopal/IB9AU-2026/blob/main/FT3_Fine_tuning_with_an_Appropriate_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Fine-tuning With an Appropriate LLM
As we saw in the previous notebook, `distilbert2` is not the base model to fine-tune for our scenario. We will use a more appropriate `distilbert_base_uncased` model for fine-tuning.




#Full Fine-tuning
We will first conduct full fine-tuning and then explore PEFT.

##Install Necessary Libraries


In [ ]:
!pip install transformers datasets
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset, DatasetDict # Import DatasetDict here
import random
import torch

##Load the Dataset:
We will load the Financial Phrasebank dataset and then take a small random sample.

In [ ]:
ds = load_dataset("warwickai/financial_phrasebank_mirror")

In [ ]:
import pandas as pd

# Convert the 'train' split of the dataset to a pandas DataFrame
df = ds["train"].to_pandas()

# Display the first 5 rows of the DataFrame
display(df.head())

In [ ]:
# Take a random sample of 400 records from the 'train' split of the dataset
# First, shuffle the dataset, then select the first 400 records
ds_train = ds["train"].shuffle(seed=42).select(range(400))

print(ds_train)
print(f"Number of records in the random sample: {len(ds_train)}")

In [ ]:
from IPython.display import Markdown

In [ ]:
display(Markdown(str(ds_train[:5])))

#Load a Tiny Pre-trained Language Model and Tokenizer

##Architecture of the Base Model

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

In [ ]:
# Print layer names
for name, param in model.named_parameters():
    print(name)

##Number of Parameters in the Base Model

In [ ]:
# Calculate the total number of parameters
total_params = sum(p.numel() for p in model.parameters())

# Print the result
print(f"Total number of parameters: {total_params}")

#Preprocess the Dataset
Now, we need to format our data for the language model. We format our data as:

sentence: `[sentence]` label: `[label]`



In [ ]:
def preprocess_function(examples):
    # Tokenize sentences directly for sequence classification
    tokenized_inputs = tokenizer(examples['sentence'], truncation=True, padding='max_length', max_length=128)
    # Assign numerical labels directly for sequence classification
    tokenized_inputs["labels"] = examples['label']
    return tokenized_inputs

tokenized_dataset = ds_train.map(preprocess_function, batched=True)


## Refine tokenized_dataset for Trainer

To prevent issues during training, we need to ensure the `tokenized_dataset` only contains columns that the model expects: `input_ids`, `attention_mask`, and `labels`. The original 'sentence' and 'label' columns are no longer needed and can cause conflicts.

In [ ]:
# Remove the original 'sentence' and 'label' columns, keeping only the tokenized inputs and the 'labels' for the model
tokenized_dataset = tokenized_dataset.remove_columns(['sentence', 'label'])

# Verify the column names after removal
print(tokenized_dataset.column_names)

##View Prepared Data

In [ ]:
for i in range(5):  # Print the first 5 examples
       example = tokenized_dataset[i]
       decoded_text = tokenizer.decode(example['input_ids'])
       print(f"Example {i + 1}:")
       print(decoded_text)
       print(example)  # Print the full dictionary for the example
       print("-" * 20)

#Setup the Fine-tuning Parameters

In [ ]:
training_args = TrainingArguments(
    output_dir="./fine_tuned_sentiment_model",
    per_device_train_batch_size=4,
    num_train_epochs=10,
    logging_dir="./logs",
    report_to="none",
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    save_strategy="epoch"
)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer, # Changed from tokenizer=tokenizer
    data_collator=data_collator
)

#Fine-tune the Model

In [ ]:
trainer.train()

#Save the Model

In [ ]:
# Define a path to save the fully fine-tuned model
save_path_full_ft = "./fully_fine_tuned_model_saved"

# Save the current state of the 'model' object
model.save_pretrained(save_path_full_ft)
tokenizer.save_pretrained(save_path_full_ft)

print(f"Fully fine-tuned model saved to: {save_path_full_ft}")

#Test the Fine-tuned Model

In [ ]:
df_train = df

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd # Ensure pandas is imported as df_train is a DataFrame

# Define the path where the fully fine-tuned model was saved
save_path_full_ft = "./fully_fine_tuned_model_saved"

# Load the fully fine-tuned model and tokenizer
# Assuming the model was saved as a SequenceClassification model with 3 labels
loaded_fine_tuned_model = AutoModelForSequenceClassification.from_pretrained(save_path_full_ft, num_labels=3)
loaded_fine_tuned_tokenizer = AutoTokenizer.from_pretrained(save_path_full_ft)

# Move the loaded model to the appropriate device (e.g., GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_fine_tuned_model.to(device)

# Ensure the model is in evaluation mode
loaded_fine_tuned_model.eval()

# Create lists to store predictions and actual labels
full_ft_predictions = []
actual_labels = []

# Iterate through the df_train dataset to make predictions
# Assuming df_train has 'sentence' and 'label' columns
for index, row in df_train.iterrows():
    sentence = row['sentence']
    actual_label = row['label']

    # Tokenize the sentence
    encoded_input = loaded_fine_tuned_tokenizer(
sentence,
        return_tensors="pt",
        truncation=True,
        padding='max_length',
        max_length=128
    )
    input_ids = encoded_input["input_ids"].to(device)
    attention_mask = encoded_input["attention_mask"].to(device)

    # Make prediction
    with torch.no_grad():
        outputs = loaded_fine_tuned_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predicted_label_id = torch.argmax(logits, dim=-1).item()

    full_ft_predictions.append(predicted_label_id)
    actual_labels.append(actual_label)

# Create a new column 'FT_label' in df_train with the predictions
df_train['FT_label'] = full_ft_predictions

print("Predictions added to df_train. Here are the first 5 entries:")
print(df_train[['sentence', 'label', 'FT_label']].head())
print("\n" + "="*50 + "\n")

# Evaluate the model
accuracy = accuracy_score(actual_labels, full_ft_predictions)
print(f"Full Fine-tuned Model Accuracy: {accuracy:.4f}")

# Map label IDs to names for the classification report
# Assuming label map is {0: 'negative', 1: 'neutral', 2: 'positive'} based on previous context
label_map_names = {0: 'negative', 1: 'neutral', 2: 'positive'}
target_names = [label_map_names[i] for i in sorted(label_map_names.keys())]

report = classification_report(
    actual_labels,
    full_ft_predictions,
    target_names=target_names,
    zero_division=0
)
print("\nFull Fine-tuned Model Classification Report:")
print(report)

#Parameter-Efficient Fine-Tuning (PEFT)
We will now explore PEFT. We will continue to use the same dataset and the same LLM.


In [ ]:
!pip install accelerate peft

In [ ]:
from peft import LoraConfig, get_peft_model
import torch

**Steps including loading and preparing the data, loading the LLM and Tokenizer remain the same as before.**

##Define Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./lora_fine_tuned_sentiment_model",
    per_device_train_batch_size=4,
    num_train_epochs=50, # Increased epochs for stronger learning
    logging_dir="./logs",
    report_to="none",
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=100,
    save_strategy="epoch"
)

In [ ]:
from transformers import AutoModelForSequenceClassification

# Reload the base model to ensure a clean application of PEFT
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS", # Changed from "CAUSAL_LM" to "SEQ_CLS"
    target_modules=["q_lin", "k_lin", "v_lin", "out_lin"], # Adjusted for DistilBERT attention layers
)

# Wrap the base model with the LoRA adapters
model = get_peft_model(model, lora_config)
model.print_trainable_parameters() # See how many parameters are trainable

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=lambda data: tokenizer.pad(data, padding='max_length', max_length=128, return_tensors='pt')
    # Call the pad method with appropriate arguments within a lambda function
)

##Train the Model

In [ ]:
trainer.train()

#Save the Model

In [ ]:
# Define a path to save the fully fine-tuned model
save_path_full_ft = "./lora_fine_tuned_model_saved"

# Save the current state of the 'model' object
model.save_pretrained(save_path_full_ft)
tokenizer.save_pretrained(save_path_full_ft)

print(f"Fully fine-tuned model saved to: {save_path_full_ft}")

##Test the Fine-tuned Model

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, PeftModel
import torch
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd # Ensure pandas is imported as df_train is a DataFrame

# Define the path where the LoRA fine-tuned model was saved
save_path_lora_ft = "./lora_fine_tuned_model_saved"

# Load the base model for sequence classification (distilbert-base-uncased)
model_name = "distilbert-base-uncased"
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# Initialize LoraConfig with the same parameters used for training
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_lin", "k_lin", "v_lin", "out_lin"],
)

# Load the LoRA adapters into the base model
loaded_lora_model = PeftModel.from_pretrained(base_model, save_path_lora_ft)

# Load the tokenizer
loaded_lora_tokenizer = AutoTokenizer.from_pretrained(save_path_lora_ft)

# Move the loaded model to the appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_lora_model.to(device)

# Ensure the model is in evaluation mode
loaded_lora_model.eval()

# Create lists to store predictions and actual labels
lora_predictions = []
actual_labels = []

# Iterate through the df_train dataset to make predictions
for index, row in df_train.iterrows():
    sentence = row['sentence']
    actual_label = row['label']

    # Tokenize the sentence
    encoded_input = loaded_lora_tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding='max_length',
        max_length=128
    )
    input_ids = encoded_input["input_ids"].to(device)
    attention_mask = encoded_input["attention_mask"].to(device)

    # Make prediction
    with torch.no_grad():
        outputs = loaded_lora_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predicted_label_id = torch.argmax(logits, dim=-1).item()

    lora_predictions.append(predicted_label_id)
    actual_labels.append(actual_label)

# Create a new column 'LORA_label' in df_train with the predictions
df_train['LORA_label'] = lora_predictions

print("LoRA Predictions added to df_train. Here are the first 5 entries:")
print(df_train[['sentence', 'label', 'LORA_label']].head())
print("\n" + "="*50 + "\n")

# Evaluate the model
accuracy = accuracy_score(actual_labels, lora_predictions)
print(f"LoRA Fine-tuned Model Accuracy: {accuracy:.4f}")

# Map label IDs to names for the classification report
label_map_names = {0: 'negative', 1: 'neutral', 2: 'positive'}
target_names = [label_map_names[i] for i in sorted(label_map_names.keys())]

report = classification_report(
    actual_labels,
    lora_predictions,
    target_names=target_names,
    zero_division=0
)
print("\nLoRA Fine-tuned Model Classification Report:")
print(report)


### Summary of LoRA vs. Full Fine-tuning Results

This summary compares the performance and parameter efficiency of the full fine-tuning approach against the LoRA (Parameter-Efficient Fine-Tuning) approach for sequence classification on the Financial Phrasebank dataset.

#### Full Fine-tuning Results:
*   **Total Parameters**: 66,955,779
*   **Accuracy**: 0.8104
*   **Classification Report (F1-Scores)**:
    *   Negative: 0.77
    *   Neutral: 0.86
    *   Positive: 0.73

#### LoRA Fine-tuning Results:
*   **Trainable Parameters**: 887,811
*   **All Parameters**: 67,843,590
*   **Trainable %**: 1.3086%
*   **Accuracy**: 0.8182
*   **Classification Report (F1-Scores)**:
    *   Negative: 0.76
    *   Neutral: 0.86
    *   Positive: 0.75

#### Performance Comparison:
Both full fine-tuning and LoRA fine-tuning achieved comparable overall accuracy on the dataset. The full fine-tuned model had an accuracy of 0.8104, while the LoRA fine-tuned model achieved a slightly higher accuracy of 0.8182. Looking at the F1-scores, both models performed similarly across the 'negative', 'neutral', and 'positive' classes, with neutral sentiment being the best predicted. The LoRA model showed a minor improvement in accuracy compared to full fine-tuning for this specific task and dataset sample.

#### Parameter Efficiency Comparison:
This is where LoRA demonstrates a significant advantage. The full fine-tuned model involved updating all 66,955,779 parameters of the `distilbert-base-uncased` model. In contrast, the LoRA fine-tuned model only updated 887,811 parameters, which constitutes a mere 1.3086% of the total parameters. This massive reduction in trainable parameters means:

*   **Reduced Computational Cost**: LoRA requires significantly less memory and computational power for training.
*   **Faster Training**: Training with fewer parameters generally leads to faster convergence and shorter training times.
*   **Smaller Model Storage**: Only the LoRA adapters (the 887,811 parameters) need to be stored, making the fine-tuned model much smaller and easier to deploy compared to storing the entire updated base model.

#### Conclusion:
For this sequence classification task on the Financial Phrasebank dataset, LoRA fine-tuning not only matched but slightly surpassed the performance of full fine-tuning, achieving a slightly higher accuracy (0.8182 vs 0.8104). Critically, it did so by training only a small fraction (1.31%) of the model's parameters. This makes LoRA an exceptionally attractive option for fine-tuning large language models, especially in resource-constrained environments, as it offers comparable (or even better) performance with vastly superior parameter efficiency.

## Summary:

### Q&A

**How did the `distilbert-base-uncased` model perform on the Financial Phrasebank dataset using both full fine-tuning and PEFT (LoRA), and what are the key comparative insights?**

Both full fine-tuning and LoRA fine-tuning achieved comparable performance, with LoRA demonstrating a slight edge in accuracy while being significantly more parameter-efficient. The full fine-tuned model achieved an accuracy of 0.8104 by updating all 66,955,779 parameters. The LoRA fine-tuned model, on the other hand, reached a slightly higher accuracy of 0.8182 while only updating 887,811 parameters, which constitutes just 1.3086\% of the total parameters. Both models exhibited similar F1-scores across negative, neutral, and positive sentiment classes, with neutral sentiment being the most accurately predicted.

### Data Analysis Key Findings

*   **LoRA Model Performance:** The LoRA fine-tuned model achieved an accuracy of 0.8182 on the `df_train` dataset.
*   **LoRA Parameter Efficiency:** LoRA fine-tuning involved updating only 887,811 trainable parameters, which is approximately 1.31% of the total model parameters (67,843,590).
*   **Full Fine-tuning Performance:** The full fine-tuned model yielded an accuracy of 0.8104.
*   **Full Fine-tuning Parameter Count:** Full fine-tuning updated all 66,955,779 parameters of the `distilbert-base-uncased` model.
*   **F1-Score Comparison:** Both models showed similar F1-scores for different sentiment classes:
    *   **Negative:** Full fine-tuning: 0.77, LoRA: 0.76
    *   **Neutral:** Full fine-tuning: 0.86, LoRA: 0.86
    *   **Positive:** Full fine-tuning: 0.73, LoRA: 0.75
*   **Overall Accuracy Comparison:** LoRA slightly outperformed full fine-tuning in accuracy (0.8182 vs. 0.8104) for this specific task.

### Insights or Next Steps

*   LoRA fine-tuning offers a highly effective method for adapting large language models to specific tasks, achieving comparable or even slightly superior performance to full fine-tuning with significantly reduced computational resources and storage requirements.
*   Given the superior parameter efficiency and competitive performance, LoRA is an ideal choice for fine-tuning in resource-constrained environments or when rapid experimentation with different models or datasets is required.


#Required Task 15
Fine-tune based on the data '`credit_card_aa.csv`'. The data is also available at https://huggingface.co/datasets/priyaannamani/credit_card_qa

Create two fine-tuned models. The first one should take `complaint` as the input and predict `policy_category`. The second should take `complaint` as the input and predict `resolution`.

Take a sample of 60 records to fine-tune and evaluate the fine-tuned models on all the records in the dataset.

You can use either or both `distilbert_base_uncased` and `distilgpt2` models.  Performance comparative analysis of how well fine-tuning works.

Write a short narrative at the end to discuss your insights from fine-tuning.